# JetRacer Semantic Lane Following - Live UI

YOLO26n-sem dùng best model engine để bám divider cam bằng các mask `road / divider / forbidden`; nhánh vật cản đã tắt. Mặc định **DISARMED**; kê bánh xe và kiểm tra chiều lái trước khi bật motor. Chạy các cell từ trên xuống.

In [1]:
import numpy as np

# Compatibility patch cho legacy dependencies trên JetPack 4
if "bool" not in np.__dict__:
    np.bool = bool

In [2]:
import sys, time, csv, threading
from pathlib import Path
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'yolo_lane_following' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from notebook3.basic_motion import JetRacerController
from yolo_lane_following.config import load_config
from yolo_lane_following.semantic_perception import YoloSemanticPerception
from yolo_lane_following.control import AdaptiveController, is_motor_command_state
from yolo_lane_following.start_gate import CompetitionStartGate, competition_motor_allowed

cfg = load_config(PROJECT_ROOT / 'yolo_lane_following/config.yaml')
engine = PROJECT_ROOT / 'yolo_lane_following/artifacts/track_yolo26n_sem_nano_fp16.engine'
if engine.exists():
    cfg['models']['semantic'] = str(engine)
    backend_name = 'TensorRT FP16'
else:
    cfg['models']['semantic'] = str(PROJECT_ROOT / 'yolo_lane_following/artifacts/track_yolo26n_sem_best.pt')
    cfg['models']['device'] = '0'
    backend_name = 'PyTorch fallback'
perception = YoloSemanticPerception(cfg)
perception.warmup()
controller = AdaptiveController(dict(cfg['control'], lane_only=True, max_lost_frames=cfg['tracking']['max_lost_frames'], lane_lock_confirm_frames=cfg['tracking'].get('lane_lock_confirm_frames', 1), lane_lock_min_confidence=cfg['tracking'].get('lane_lock_min_confidence', 0.0)))
start_gate = CompetitionStartGate(cfg['competition'])
print('Semantic backend:', backend_name)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Loading /workspace/jetracer_official/yolo_lane_following/artifacts/track_yolo26n_sem_nano_fp16.engine for TensorRT inference...
[08/20/2026-04:45:13] [TRT] [I] [MemUsageChange] Init CUDA: CPU +229, GPU +0, now: CPU 294, GPU 2514 (MiB)
[08/20/2026-04:45:13] [TRT] [I] Loaded engine size: 4 MiB
[08/20/2026-04:45:13] [TRT] [W] Using an engine plan file across different models of devices is not recommended and is likely to affect performance or even cause errors.
[08/20/2026-04:45:18] [TRT] [W] TensorRT was linked against cuBLAS/cuBLAS LT 10.2.3 but loaded cuBLAS/cuBLAS LT 10.2.2
[08/20/2026-04:45:18] [TRT] [I] [MemUsageChange] Init cuBLAS/cuBLASLt: CPU +155, GPU +241, now: CPU 462, GPU 2768 (MiB)
[08/20/2026-04:45:26] [TRT] [I] [MemUsageChange] Init cuDNN: CPU +241, GPU +350, now: CPU 703, GPU 3118 (MiB)
[08/20/2026-04:45:26] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in engine deserialization: CPU +0, GPU +3, now: CPU 0, GPU 3 (MiB)
[08/20/2026-04:45:26] [TRT] [W] TensorRT was

Nếu camera CSI bị khóa, chạy `sudo systemctl restart nvargus-daemon` trong terminal trước cell tiếp theo. Notebook không nhúng mật khẩu sudo.

In [3]:
try:
    camera.running = False; camera.unobserve_all()
except Exception:
    pass
camera = CSICamera(width=224, height=224, capture_fps=60)
car = JetRacerController(cfg['control']['steering_gain'], cfg['control']['steering_offset'], cfg['control']['throttle_gain'], cfg['control']['throttle_max'])
car.stop(); car.center_steering()
snapshot = None


GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3264 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3264 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 120.000005 fps Duration = 8333333 ; Analog Gain range min 1.000000, max 10.625000; Exposur

[ WARN:0@82.695] global /tmp/opencv/modules/videoio/src/cap_gstreamer.cpp (1405) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!


In [4]:
state = widgets.ToggleButtons(options=['stop','live'], value='stop', description='State')
armed = widgets.Checkbox(value=False, description='ARM MOTOR')
competition_mode = widgets.Checkbox(value=False, description='COMPETITION')
# max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=0.0, max=0.5, step=0.005, description='Max throttle')
# steering_scale = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='Steer scale')
# max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=cfg['control']['throttle_limit_min'], max=cfg['control']['throttle_limit_max'], step=0.005, description='Max throttle')
max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=cfg['control']['throttle_limit_min'], max=2, step=0.005, description='Max throttle')
steering_scale = widgets.FloatSlider(value=1.0, min=0.75, max=4, step=0.05, description='Steer scale')
raw_view = widgets.Image(format='jpeg', width=224, height=224)
debug_view = widgets.Image(format='jpeg', width=224, height=224)
status = widgets.HTML(value='<b>STOPPED / DISARMED</b>')
blank = np.zeros((224,224,3), np.uint8); raw_view.value=bgr8_to_jpeg(blank); debug_view.value=bgr8_to_jpeg(blank)
display(widgets.VBox([widgets.HBox([raw_view, debug_view]), status, widgets.HBox([state, armed, competition_mode]), max_throttle, steering_scale]))

In [5]:
callback_lock = threading.Lock(); last_tick = time.perf_counter(); fps_ema = 0.0; frame_index = 0
log_dir = PROJECT_ROOT / 'yolo_lane_following/logs'; log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / time.strftime('semantic_live_%Y%m%d_%H%M%S.csv')
log_stream = log_path.open('w', newline=''); log_writer = csv.writer(log_stream); log_buffer=[]
log_writer.writerow(['timestamp','fps','lane_confidence','target_x','risk','white_front','steering','throttle','throttle_max','state','armed','competition','green','start_authorized','start_latency_ms'])

def live_update(change):
    global last_tick, fps_ema, frame_index
    if state.value != 'live' or not callback_lock.acquire(False): return
    try:
        frame = change['new']; now = time.perf_counter(); dt=max(0.005, now-last_tick); last_tick=now
        green = start_gate.update(frame) if competition_mode.value else None
        result = perception.infer(frame)
        selected_max = controller.set_throttle_limit(float(max_throttle.value))
        car.max_throttle = selected_max
        command = controller.update(result.lane, 0.0, frame.shape[1], dt,
                                    result.forbidden_left, result.forbidden_right, result.escape_steering, result.forbidden_front)
        command.steering = float(np.clip(command.steering * steering_scale.value, -1, 1))
        command.throttle = float(np.clip(command.throttle, -selected_max, selected_max))
        safe_state = is_motor_command_state(command.state)
        motor_allowed = competition_motor_allowed(armed.value, competition_mode.value, start_gate.authorized, safe_state)
        if motor_allowed:
            car.set_steering(command.steering); car.set_throttle(command.throttle)
        else:
            car.stop(); car.center_steering()
        instant=1.0/dt; fps_ema=instant if fps_ema==0 else .2*instant+.8*fps_ema
        rendered=result.annotated.copy(); cv2.putText(rendered, command.state, (5,16), cv2.FONT_HERSHEY_SIMPLEX, .45, (0,255,255), 1, cv2.LINE_AA)
        if green is not None and green.detected: cv2.circle(rendered, green.center, green.radius, (0,255,0), 2)
        gate_text = 'START' if (not competition_mode.value or start_gate.authorized) else 'WAIT GREEN'
        gate_latency = start_gate.authorization_latency_ms if start_gate.authorization_latency_ms is not None else -1.0
        cv2.putText(rendered, gate_text, (5,34), cv2.FONT_HERSHEY_SIMPLEX, .45, (0,255,0) if gate_text == 'START' else (0,165,255), 1, cv2.LINE_AA)
        frame_index += 1
        if frame_index % int(cfg['runtime'].get('display_stride', 3)) == 0:
            raw_view.value=bgr8_to_jpeg(frame); debug_view.value=bgr8_to_jpeg(rendered)
            status.value='<b>%s | %s %.1f ms | %s | FPS %.1f | conf %.2f | risk %.2f | white-front %.2f | steer %+.3f | throttle %.3f / max %.3f</b>' % (backend_name, gate_text, gate_latency, command.state, fps_ema, result.lane.confidence, result.obstacle_risk, result.forbidden_front, command.steering, command.throttle, selected_max)
        log_buffer.append([time.time(), fps_ema, result.lane.confidence, result.lane.target_x, result.obstacle_risk, result.forbidden_front, command.steering, command.throttle, selected_max, command.state, int(armed.value), int(competition_mode.value), int(green.detected) if green is not None else 0, int(start_gate.authorized), gate_latency])
        if len(log_buffer) >= 20: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
    except Exception as exc:
        car.stop(); car.center_steering(); state.value='stop'; status.value='<b style="color:red">ERROR: %s</b>' % exc
    finally:
        callback_lock.release()

def safety_changed(change):
    if change.get('new') == 'stop':
        car.stop(); car.center_steering()
        if change.get('new') == 'stop': start_gate.reset()
        if change.get('new') == 'stop' and log_buffer: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
def armed_changed(change):
    if change.get('new'):
        state.value = 'live'
    else:
        state.value = 'stop'; car.stop(); car.center_steering()
def competition_changed(change):
    start_gate.reset(); car.stop(); car.center_steering()
state.observe(safety_changed, names='value'); armed.observe(armed_changed, names='value'); competition_mode.observe(competition_changed, names='value')
camera.observe(live_update, names='value'); camera.running=True
print('Camera running. Chọn live để infer; ARM MOTOR chỉ sau khi kê bánh và kiểm tra mask trắng.')

Camera running. Chọn live để infer; ARM MOTOR chỉ sau khi kê bánh và kiểm tra mask trắng.
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
[JetRacer] Đã

## Dừng an toàn - luôn chạy cell này trước khi đóng notebook

In [ ]:
#state.value='stop'; armed.value=False; car.stop(); car.center_steering()
#camera.running=False; camera.unobserve_all()
#if log_buffer: log_writer.writerows(log_buffer); log_buffer.clear()
#log_stream.flush(); log_stream.close()
#print('Stopped safely. Log:', log_path)